# HuggingFace Pipelines: High-Level Inference

Reach for this when you need: 
- Reference for rapid prototyping and deployment with pre-made pipelines.
- To implement task-specific inference (Summarization, QA, NER).
- Reference for batch processing and device placement.

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

## 1. Core Task Pipelines

| Task | API | Logic |
| :--- | :--- | :--- |
| Classification | `sentiment-analysis` | Binary/Multi-label logits to labels |
| NER | `token-classification` | Extraction of entities (Names, Orgs) |
| Summarization | `summarization` | Seq2Seq generation to shorten text |
| QA | `question-answering` | Extraction of answer spans from context |

In [ ]:
# Sentiment analysis pipeline
classifier = pipeline("sentiment-analysis", device=device)
result = classifier("PyTorch pipelines are incredibly productive for production.")

# NER pipeline (Entity Extraction)
ner = pipeline("ner", grouped_entities=True, device=device)
ner_res = ner("Sundar Pichai is the CEO of Google in Mountain View.")

## 2. Model Switching

You can specify ANY model from the HuggingFace Hub directly in the pipeline interface.

✅ **Use when**: Standard models aren't accurate enough for your specific domain (e.g. BioBERT).
❌ **Don't use when**: You need custom preprocessing logic before inference.

In [ ]:
# Loading a specific domain-adapted model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=device)

text = """ 
The Transformer is a deep learning model that adopts the mechanism of attention, 
weighing the significance of each part of the input data. It is used primarily 
in the field of natural language processing (NLP).
"""
summary = summarizer(text, max_length=50, min_length=20, do_sample=False)

### Common Pitfalls
- **Batch Size**: Pipelines process one item at a time by default. For large datasets, use the `batch_size` argument to increase GPU throughput.
- **VRAM Management**: Multiple pipelines on the same GPU will consume memory cumulatively; delete old ones if switching tasks.
- **Device Index**: PyTorch `device=torch.device('cuda')` is different from HF `device=0`. Ensure you use the integer index for HF pipelines.

### Key Takeaways
- HuggingFace `pipeline()` handles tokenization, forward pass, and decoding in a single call.
- Always use `device=0` (or the appropriate GPU index) to avoid slow CPU inference.
- `grouped_entities=True` is critical for NER if you want full words instead of subwords (##tokens).